In [ ]:
import pandas as pd
import numpy as np
import os


from matplotlib import pyplot as plt
import sklearn.ensemble

from sklearn import preprocessing
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
import graphviz

from sklearn.tree import DecisionTreeClassifier
from sklearn import tree

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler

from sklearn.decomposition import PCA

from sklearn.model_selection import cross_val_score
from sklearn import metrics
from sklearn.metrics import accuracy_score
from sklearn.metrics import roc_curve, auc

from anchor import utils
from anchor import anchor_tabular

from xailib.data_loaders.dataframe_loader import prepare_dataframe

from xailib.explainers.lime_explainer import LimeXAITabularExplainer
from xailib.explainers.lore_explainer import LoreTabularExplainer
from xailib.explainers.shap_explainer_tab import ShapXAITabularExplainer

from xailib.models.sklearn_classifier_wrapper import sklearn_classifier_wrapper

from plot_explanation import PlotExplanation


In [ ]:
path = os.getcwd()
print(path)
source_file=f'../../datasets/mushrooms.csv'

In [ ]:
class_field = 'class'
df=pd.read_csv(source_file)
df.head(10)

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
y_label = df['class'].unique()

In [ ]:
y_label

In [ ]:
df.shape

In [ ]:
df, feature_names, class_values, numeric_columns, rdf, real_feature_names, features_map = prepare_dataframe(df, class_field)

### Data description

In [ ]:
df.describe()

### Cardinality

In [ ]:
count_df = pd.DataFrame()
for col in df.columns:
    column_counts = df[col].value_counts().reset_index()
    column_counts.columns = ['Category', f'Count_{col}']
    count_df = pd.concat([count_df, column_counts], axis=1)
count_df.head(20)

## The dataset has values in strings. We need to convert all the unique values to integers. Perform label encoding on the data.

In [ ]:
labelencoder=LabelEncoder()
for col in df.columns:
    df[col] = labelencoder.fit_transform(df[col])
 
df.head()

In [ ]:
df['class'].unique()

In [ ]:
print(df.groupby('class').size())

In [ ]:
df.corr()

### Standardising the data

In [ ]:
scaler = MinMaxScaler()

In [ ]:
col_name = df.columns.values
col_name = list(col_name)

In [ ]:
print(col_name)

In [ ]:
df_scaled = scaler.fit_transform(df.to_numpy())
df_scaled = pd.DataFrame(df_scaled, columns=col_name)
 
print("Scaled Dataset Using MinMaxScaler")
df_scaled.head()

### Non super utile

In [ ]:
X = df.iloc[:,1:23]  # all rows, all the features and no labels
y = df.iloc[:, 0]  # all rows, label only
X.head()
y.head()

In [ ]:
scaler = StandardScaler()
X=scaler.fit_transform(X)
X

### PCA

In [ ]:
pca = PCA()
pca.fit_transform(X)

In [ ]:
covariance=pca.get_covariance()

In [ ]:
explained_variance=pca.explained_variance_
explained_variance

In [ ]:
with plt.style.context('dark_background'):
    plt.figure(figsize=(6, 4))
    
    plt.bar(range(22), explained_variance, alpha=0.5, align='center',
            label='individual explained variance')
    plt.ylabel('Explained variance ratio')
    plt.xlabel('Principal components')
    plt.legend(loc='best')
    plt.tight_layout()

### Split into train and test

In [ ]:
X = df.drop('class', axis=1)  # Features (all columns except 'target_class')
y = df['class']

In [ ]:
y.head()

In [ ]:
test_size = 0.8
random_state = 4
X_train, X_test, y_train, y_test = train_test_split(X,y,
                                                        test_size=test_size,
                                                        random_state=random_state
                                                    )

## SINGLE DECISION TREE

In [ ]:
dt_gini = DecisionTreeClassifier(criterion='gini', max_depth=3, random_state=0)


# fit the model
dt_gini.fit(X_train, y_train)

In [ ]:
y_pred_gini = dt_gini.predict(X_test)

In [ ]:
y_prob = dt_gini.predict_proba(X_test)[:,1] # This will give you positive class prediction probabilities  
y_pred = np.where(y_prob > 0.5, 1, 0) # This will threshold the probabilities to give class predictions.
dt_gini.score(X_test, y_pred)

In [ ]:
confusion_matrix=metrics.confusion_matrix(y_test,y_pred)
confusion_matrix

In [ ]:
# print the scores on training and test set

print('Training set score: {:.4f}'.format(dt_gini.score(X_train, y_train)))

print('Test set score: {:.4f}'.format(dt_gini.score(X_test, y_test)))

In [ ]:
df['class']

In [ ]:
fig = plt.figure(figsize=(25,20))
_ = tree.plot_tree(dt_gini, 
                   feature_names=X.columns.values,  
                   class_names=str(y_label),
                   filled=True)

### Train a Random Forest

Train the model on the training set. Once the model has been learned, we use a wrapper class to get access to the model for XAI lib

In [ ]:
bb = RandomForestClassifier(n_estimators=1, random_state=random_state)
bb.fit(X_train, y_train)
bbox = sklearn_classifier_wrapper(bb)

### Controllo la curva AOC (molto male)

In [ ]:
y_prob = bb.predict_proba(X_test)[:,1] # This will give you positive class prediction probabilities  
y_pred = np.where(y_prob > 0.5, 1, 0) # This will threshold the probabilities to give class predictions.
bb.score(X_test, y_pred)

In [ ]:
confusion_matrix=metrics.confusion_matrix(y_test,y_pred)
confusion_matrix

In [ ]:
auc_roc=metrics.roc_auc_score(y_test,y_pred)
auc_roc

In [ ]:
false_positive_rate, true_positive_rate, thresholds = roc_curve(y_test, y_prob)
roc_auc = auc(false_positive_rate, true_positive_rate)
roc_auc

In [ ]:
%matplotlib inline
plt.figure(figsize=(10,10))
plt.title('Receiver Operating Characteristic')
plt.plot(false_positive_rate,true_positive_rate, color='red',label = 'AUC = %0.2f' % roc_auc)
plt.legend(loc = 'lower right')
plt.plot([0, 1], [0, 1],linestyle='--')
plt.axis('tight')
plt.ylabel('True Positive Rate')
plt.xlabel('False Positive Rate')

### Train a Logistic Regression

In [ ]:
df_scaled
X_r = df_scaled.drop('class', axis=1)  # Features (all columns except 'target_class')
y_r = df_scaled['class']

In [ ]:
test_size = 0.4
random_state = 4
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_r,y_r,
                                                        test_size=test_size,
                                                        random_state=random_state,
                                                        stratify=y_r)

In [ ]:
model_LR= LogisticRegression()

In [ ]:
model_LR.fit(X_train_r, y_train_r)

In [ ]:
y_prob_r = model_LR.predict_proba(X_test_r)[:,1] # This will give you positive class prediction probabilities  
y_pred_r = np.where(y_prob_r > 0.5, 1, 0) # This will threshold the probabilities to give class predictions.
model_LR.score(X_test_r, y_pred_r)

In [ ]:
confusion_matrix=metrics.confusion_matrix(y_test_r,y_pred_r)
confusion_matrix

In [ ]:
false_positive_rate, true_positive_rate, thresholds = roc_curve(y_test, y_prob)
roc_auc = auc(false_positive_rate, true_positive_rate)
roc_auc

In [ ]:
%matplotlib inline
plt.figure(figsize=(10,10))
plt.title('Receiver Operating Characteristic')
plt.plot(false_positive_rate,true_positive_rate, color='red',label = 'AUC = %0.2f' % roc_auc)
plt.legend(loc = 'lower right')
plt.plot([0, 1], [0, 1],linestyle='--')
plt.axis('tight')
plt.ylabel('True Positive Rate')
plt.xlabel('False Positive Rate')

### TRAIN a XGBoost

In [ ]:
clf = GradientBoostingClassifier(n_estimators=100, learning_rate=1.0,
max_depth=1, random_state=random_state).fit(X_train, y_train)
clf.score(X_test, y_test)

In [ ]:
y_prob = clf.predict_proba(X_test)[:,1] # This will give you positive class prediction probabilities  
y_pred = np.where(y_prob > 0.5, 1, 0) # This will threshold the probabilities to give class predictions.
clf.score(X_test, y_pred)

In [ ]:
confusion_matrix=metrics.confusion_matrix(y_test,y_pred)
confusion_matrix

In [ ]:
false_positive_rate, true_positive_rate, thresholds = roc_curve(y_test, y_prob)
roc_auc = auc(false_positive_rate, true_positive_rate)
roc_auc

In [ ]:
%matplotlib inline
plt.figure(figsize=(10,10))
plt.title('Receiver Operating Characteristic')
plt.plot(false_positive_rate,true_positive_rate, color='red',label = 'AUC = %0.2f' % roc_auc)
plt.legend(loc = 'lower right')
plt.plot([0, 1], [0, 1],linestyle='--')
plt.axis('tight')
plt.ylabel('True Positive Rate')
plt.xlabel('False Positive Rate')

### LORE

Select a new instance to be classfied by the model and print the predicted class.

In [ ]:
r=list(range(0,6))
for i in r:
    inst = X_train.iloc[i].values
    print('Instance ',inst)
    print('True class ',y_train.iloc[6])
    print('Predicted class ',bb.predict(inst.reshape(1, -1)))
    explainer = LoreTabularExplainer(bbox)
    config = {'neigh_type':'geneticp', 'size':1000, 'ocr':0.1, 'ngen':10}
    explainer.fit(df, class_field, config)

    exp = explainer.explain(inst)
    print(exp)
    exp.expDict

In [ ]:
exp.plotRules()

In [ ]:
exp.plotCounterfactualRules()

### SHAP

In [ ]:
explainer = ShapXAITabularExplainer(bbox, feature_names)
config = {'explainer' : 'tree', 'X_train' : X_train[0:300]}
explainer.fit(config)

In [ ]:
exp_s = explainer.explain(inst)
print(exp)

In [ ]:
result = exp_s.getFeaturesImportance()

In [ ]:
result

In [ ]:
pe = PlotExplanation(feature_names=feature_names,
            real_feature_names=real_feature_names,
            numeric_columns=numeric_columns,
            instance_number=X_train.iloc[1].values,
            x_train=X_train,
            expDict= exp.expDict,
            feature_importance_type='Shap',
            feature_importance=result)

In [ ]:
numeric_columns

In [ ]:
df_v = pe.prepare_dataframe()

### ANCHOR
https://github.com/marcotcr/anchor/blob/master/notebooks/Anchor%20on%20tabular%20data.ipynb


Per ANCHOR bisogna formattare il dataset in modo che ci sia:
- class_names: una lista coi valori della classe da predirre
- feature_names: lista dei nomi delle colonne
- train: array dei dati di training
- test: array dei dati di test
- labels_test: array con del y_test
- categorical_names: un dizionario che ha come key il numero della colonna categorica e come value la lista delle categorie

In [ ]:
class_field = 'class'
df_a=pd.read_csv(source_file)
df_a.head(10)

In [ ]:
X = df_a.drop('class', axis=1)  # Features (all columns except 'class')
y = df_a['class']  # Target class

In [ ]:
class_names = list(df_a['class'].unique())
feature_names = list(X.columns)
    # Initialize an empty dictionary to store the results
categorical_names = {}
    # Loop through the columns of the DataFrame
for col_num, col_name in enumerate(X.columns):
    if X[col_name].dtype == 'object':  # Check if the column is categorical
        unique_categories = X[col_name].unique().tolist()
        categorical_names[col_num] = unique_categories
    categorical_names

In [ ]:
labelencoder=LabelEncoder()
for col in X.columns:
    X[col] = labelencoder.fit_transform(X[col])
X.head()
y = labelencoder.fit_transform(y)


In [ ]:
categorical_names

In [ ]:
test_size = 0.4
random_state = 4
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                        test_size=test_size,
                                                        random_state=random_state,
                                                        stratify=y)

In [ ]:
c = sklearn.ensemble.RandomForestClassifier(n_estimators=50, n_jobs=5)
c.fit(X_train.values, y_train)
print('Train', sklearn.metrics.accuracy_score(y_train, c.predict(X_train.values)))
print('Test', sklearn.metrics.accuracy_score(y_test, c.predict(X_test.values)))

In [ ]:
explainer = anchor_tabular.AnchorTabularExplainer(
    class_names,
    feature_names,
    X_train.values,
    categorical_names)

In [ ]:
feature_names

In [ ]:
explainer.class_names

In [ ]:
idx = 6
np.random.seed(1)
print('Prediction: ', explainer.class_names[c.predict(X_test.values[idx].reshape(1, -1))[0]])
exp = explainer.explain_instance(X_test.values[idx], c.predict, threshold=0.95)

In [ ]:
print('Anchor: %s' % (' AND '.join(exp.names())))
print('Precision: %.2f' % exp.precision())
print('Coverage: %.2f' % exp.coverage())

In [ ]:
# Get test examples where the anchora pplies
fit_anchor = np.where(np.all(X_test.values[:, exp.features()] == X_test.values[idx][exp.features()], axis=1))[0]
print('Anchor test precision: %.2f' % (np.mean(c.predict(X_test.values[fit_anchor]) == c.predict(X_test.values[idx].reshape(1, -1)))))
print('Anchor test coverage: %.2f' % (fit_anchor.shape[0] / float(X_test.values.shape[0])))

In [ ]:
exp.show_in_notebook()

### Struttura per il json

In [ ]:
import json
import os

In [ ]:
plot_explanation

In [ ]:
#data descriptio

df_range=pd.concat(
    {'min':X_train.min(),
     'max':X_train.max(),
     'std':X_train.std(),
     'q1':X_train.quantile(0.25),
     'median':X_train.quantile(0.50),
     'q3':X_train.quantile(0.75),
     },axis=1)

### Esporto_json per D3

In [ ]:
def parse_dict_first(lore_obj): #creo un dizionario dell'istanza, alcuni oggetti però vanno parsati meglio
    istance = lore_obj.__dict__
    return istance

def update_object(istance):
    istance.update(
    {'rule': rule_list,
     'crules':counterrule_list,
     'exemplars':exemplars_list,
     'dt':dt_dict,
     'distribution':data_desc,
     'inst':inst_dict,
     'fi_shap':fi_shap
    })
    return istance   

In [ ]:
class NpEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return json.JSONEncoder.default(self, obj)

In [ ]:
filename = "../JSON_istance/istance_5.json"
with open(filename, "w") as file:
    json.dump(istance_5, file, cls=NpEncoder)